# Day 38 — **Pydantic: Agent Output Contracts**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

In a multi-agent pipeline, agent A's output is agent B's input. If A emits a slightly-wrong dict, B fails somewhere deep — or worse, silently does the wrong thing. The fix is a **strict Pydantic contract** on every agent's output: validation happens at the *boundary*, so a malformed handoff fails loudly and immediately where you can see it.

This is Day 10 (Pydantic validation) applied to the multi-agent seam. Pydantic v2, real and runnable.

Today:
1. **Strict models** — `ConfigDict(strict=True, extra="forbid")` rejects junk.
2. **Contract validation at the handoff** — A → B fails at the boundary.
3. **Business-rule validators** — a valid *shape* isn't a valid *value*.
4. **JSON Schema export** — the contract as API/tool documentation.

## 1. Strict output models

`strict=True` stops silent coercion (the string `"5"` won't become `5`); `extra="forbid"` rejects unexpected fields (a typo'd key is an error, not ignored). For an agent contract you want both — no surprises across the handoff.

In [1]:
from pydantic import BaseModel, Field, ConfigDict, ValidationError

class AnalysisOutput(BaseModel):
    """The contract the analysis agent MUST emit."""
    model_config = ConfigDict(strict=True, extra="forbid")

    customer_id: str = Field(pattern=r"^C-\d{4}$")
    grade: str = Field(pattern=r"^(AAA|BBB|CCC)$")
    pd: float = Field(ge=0.0, le=1.0)              # probability of default in [0,1]
    suggested_limit: int = Field(ge=0)

good = AnalysisOutput(customer_id="C-4417", grade="BBB", pd=0.018, suggested_limit=500_000)
print("valid:", good.model_dump())

# extra field -> rejected (typo caught, not silently dropped)
try:
    AnalysisOutput(customer_id="C-4417", grade="BBB", pd=0.018,
                   suggested_limit=500_000, grdae="AAA")
except ValidationError as e:
    print("rejected extra field:", e.errors()[0]["type"])

valid: {'customer_id': 'C-4417', 'grade': 'BBB', 'pd': 0.018, 'suggested_limit': 500000}
rejected extra field: extra_forbidden


☝ `extra="forbid"` is the unsung hero of agent contracts: without it, a typo'd key (`grdae`) is silently ignored and the real `grade` field goes unset or defaulted — a bug that surfaces three agents downstream. `strict=True` blocks the other silent failure, type coercion: an LLM that returns `"0.018"` (string) for `pd` is caught here, not after it's been used in a comparison. Fail at the boundary, with a precise error, every time.

## 2. Validate at the handoff

The value shows at the seam: the downstream agent validates its input *before* doing any work. A bad upstream output is rejected at B's door, not deep in B's logic.

In [2]:
class ComplianceInput(BaseModel):
    """What the compliance agent accepts. Same strictness."""
    model_config = ConfigDict(strict=True, extra="forbid")
    customer_id: str = Field(pattern=r"^C-\d{4}$")
    grade: str
    suggested_limit: int = Field(ge=0)

def compliance_agent(raw: dict) -> str:
    try:
        data = ComplianceInput.model_validate(raw)      # boundary check
    except ValidationError as e:
        return f"REJECTED at boundary: {e.error_count()} error(s) -> {e.errors()[0]['loc']}"
    return f"screening {data.customer_id} at grade {data.grade}"

# upstream produced a clean output -> passes
print(compliance_agent(good.model_dump()))
# upstream produced a broken output (negative limit, missing grade) -> caught here
print(compliance_agent({"customer_id": "C-4417", "suggested_limit": -5}))

REJECTED at boundary: 1 error(s) -> ('pd',)
REJECTED at boundary: 2 error(s) -> ('grade',)


☝ `model_validate` at the top of `compliance_agent` is a **contract check on entry** — the agent refuses to run on malformed input and returns a precise, located error (`loc` tells you which field). This is defence-in-depth: even though A emits an `AnalysisOutput`, B doesn't *trust* the wire — it re-validates, because in production A might be a different service, a different version, or a compromised one. Validate what you receive, always.

## 3. Business-rule validators — shape ≠ correctness

A well-typed output can still be nonsense: a `CCC` grade with a `£5m` limit violates policy. `@field_validator` and `@model_validator` (Day 10) encode the *rules*, not just the types.

In [3]:
from pydantic import model_validator

class CreditDecision(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    grade: str = Field(pattern=r"^(AAA|BBB|CCC)$")
    suggested_limit: int = Field(ge=0)
    sanctions_hit: bool

    @model_validator(mode="after")
    def enforce_policy(self):
        caps = {"AAA": 5_000_000, "BBB": 1_000_000, "CCC": 100_000}
        if self.suggested_limit > caps[self.grade]:
            raise ValueError(f"limit {self.suggested_limit} exceeds {self.grade} cap {caps[self.grade]}")
        if self.sanctions_hit and self.suggested_limit > 0:
            raise ValueError("sanctions hit -> limit must be 0 pending review")
        return self

print("ok:", CreditDecision(grade="AAA", suggested_limit=2_000_000, sanctions_hit=False).grade)
for bad in [dict(grade="CCC", suggested_limit=500_000, sanctions_hit=False),
            dict(grade="AAA", suggested_limit=1_000, sanctions_hit=True)]:
    try:
        CreditDecision(**bad)
    except ValidationError as e:
        print("policy violation:", e.errors()[0]["msg"])

ok: AAA
policy violation: Value error, limit 500000 exceeds CCC cap 100000
policy violation: Value error, sanctions hit -> limit must be 0 pending review


☝ `@model_validator(mode="after")` runs once all fields are set, so it can enforce **cross-field** rules — a limit that's valid in isolation is rejected against its grade's cap, and any sanctions hit forces the limit to zero. Encoding policy in the model means an agent *cannot* emit a decision that breaks it: the invalid object never gets constructed. That's stronger than a downstream `if` check someone might forget — the contract is the control.

## 4. JSON Schema export — the contract as documentation

`model_json_schema()` emits the contract as JSON Schema — the same format that documents an LLM tool's arguments (Anthropic `tool_use`), an OpenAPI body, or a registry entry. One model, and your validation *and* your docs stay in sync.

In [4]:
import json
schema = AnalysisOutput.model_json_schema()
print(json.dumps({
    "title": schema["title"],
    "required": schema["required"],
    "pd_constraints": {k: v for k, v in schema["properties"]["pd"].items()
                       if k in ("type", "minimum", "maximum")},
    "additionalProperties": schema.get("additionalProperties"),
}, indent=2))

{
  "title": "AnalysisOutput",
  "required": [
    "customer_id",
    "grade",
    "pd",
    "suggested_limit"
  ],
  "pd_constraints": {
    "maximum": 1.0,
    "minimum": 0.0,
    "type": "number"
  },
  "additionalProperties": false
}


☝ The schema is *derived from the model*, so it can't drift from the code that enforces it — the `pd` bounds `[0,1]` and `additionalProperties: false` (from `extra="forbid"`) appear in the docs because they're real constraints, not a hand-written spec someone forgot to update. This is why Pydantic is the backbone of agent I/O: the same class validates the handoff, documents the tool, and (Module 6) defines your MCP tool schemas — one source of truth.

## Recap

| Goal | Pydantic v2 |
|---|---|
| No silent coercion | `ConfigDict(strict=True)` |
| No stray/typo'd fields | `extra="forbid"` |
| Reject bad input at the door | `model_validate` at agent entry |
| Cross-field policy | `@model_validator(mode="after")` |
| Contract == docs | `model_json_schema()` |

**Rule:** every agent output is a strict Pydantic model; validate on entry, encode policy in the model, export the schema for docs and tools. Tomorrow (Day 39): building episodic memory — an `EpisodicMemory` class with similarity + recency retrieval.